# Extracting Metadata from Jupyter Notebooks

This notebook demonstrates a practical extractor script that scans `.ipynb` files (which are JSON),
extracts structured experiment metadata and measurement tables, and exports them to CSV files.

**Note:** All code cells below are examples. Do not run them here.

## Overview of the Extractor Approach

The extractor script follows this workflow:

1. **Scan** a folder for all `.ipynb` files.
2. **Parse** each notebook as JSON.
3. **Find** a metadata cell (e.g., a code cell containing a JSON variable like `experiment_metadata`).
4. **Extract** the metadata object and any tabular outputs (or JSON blobs) from other cells.
5. **Normalize** nested structures (like a list of measurements) into flat CSV rows using `pd.json_normalize()`.
6. **Save** to CSV files for downstream analysis, database import, or ELN integration.

This is non-destructive—the original notebooks remain unchanged.

## Example: a simple experiment notebook

Imagine an experiment notebook with this structure:

- **Metadata cell** (code): defines `experiment_metadata` as a dict with experiment ID, date, operator, etc.
- **Data cell** (code): runs an analysis and produces output (or stores results in a dict).

Example notebook cell (what the metadata cell might contain):

```python
experiment_metadata = {
    'experiment_id': 'exp-UV-001',
    'date': '2026-02-06',
    'operator': 'Alice Chen',
    'project': 'UV-Absorption-Study',
    'description': 'Measuring absorbance of dye solutions at 280 nm',
    'measurements': [
        {'sample': 'control', 'concentration': 0.0, 'absorbance': 0.05, 'wavelength_nm': 280},
        {'sample': 'dye_A', 'concentration': 0.001, 'absorbance': 0.23, 'wavelength_nm': 280},
        {'sample': 'dye_B', 'concentration': 0.001, 'absorbance': 0.18, 'wavelength_nm': 280},
    ]
}
```

## Extractor Script (Example Implementation)

In [ ]:
# File: extract_notebook_metadata.py
# A simple extractor to pull experiment metadata from .ipynb files
# Usage (do not run here): python extract_notebook_metadata.py --input-dir ./notebooks --output-dir ./extracted_data

import json
import re
import argparse
from pathlib import Path
from typing import Any, Dict, List, Optional
import pandas as pd


def extract_metadata_from_cell(source: List[str]) -> Optional[Dict[str, Any]]:
    """
    Try to extract a JSON-like dict named 'experiment_metadata' from Python source code.
    Returns the dict if found, else None.
    """
    code = ''.join(source)
    # Simple regex to find: experiment_metadata = {...}
    match = re.search(r'experiment_metadata\s*=\s*(\{.*?\})(?=\n|$)', code, re.DOTALL)
    if match:
        try:
            # Evaluate as a Python literal (safe for well-formed dicts)
            return eval(match.group(1))
        except Exception as e:
            print(f"Warning: Could not parse metadata dict: {e}")
    return None


def process_notebook(notebook_path: Path) -> Optional[Dict[str, Any]]:
    """
    Open a .ipynb file, scan all cells for experiment_metadata, and return it.
    """
    try:
        with notebook_path.open('r', encoding='utf-8') as f:
            nb = json.load(f)
    except Exception as e:
        print(f"Error reading {notebook_path}: {e}")
        return None

    cells = nb.get('cells', [])
    for cell in cells:
        if cell.get('cell_type') == 'code':
            source = cell.get('source', [])
            metadata = extract_metadata_from_cell(source)
            if metadata:
                # Add notebook filename to the record for traceability
                metadata['_source_notebook'] = notebook_path.name
                return metadata
    return None


def export_to_csv(data: Dict[str, Any], output_dir: Path, notebook_name: str) -> None:
    """
    Export experiment metadata and nested lists (like 'measurements') to CSV.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Check if there are nested lists (e.g., 'measurements')
    for key, value in data.items():
        if isinstance(value, list) and value and isinstance(value[0], dict):
            # Normalize this list into a DataFrame
            df = pd.json_normalize(data, key)
            csv_name = f"{notebook_name}_{key}.csv"
            csv_path = output_dir / csv_name
            df.to_csv(csv_path, index=False)
            print(f"Exported: {csv_path}")
    
    # Also export top-level non-list metadata
    scalar_meta = {k: v for k, v in data.items() if not isinstance(v, list)}
    if scalar_meta:
        meta_csv = output_dir / f"{notebook_name}_metadata.csv"
        pd.DataFrame([scalar_meta]).to_csv(meta_csv, index=False)
        print(f"Exported: {meta_csv}")


def main():
    parser = argparse.ArgumentParser(
        description='Extract experiment metadata from Jupyter notebooks'
    )
    parser.add_argument(
        '--input-dir', type=Path, default=Path('.'),
        help='Directory containing .ipynb files'
    )
    parser.add_argument(
        '--output-dir', type=Path, default=Path('./extracted_data'),
        help='Directory to save extracted CSV files'
    )
    args = parser.parse_args()
    
    input_dir = args.input_dir
    output_dir = args.output_dir
    
    print(f"Scanning {input_dir} for .ipynb files...")
    notebooks = list(input_dir.glob('**/*.ipynb'))
    print(f"Found {len(notebooks)} notebook(s)")
    
    for nb_path in notebooks:
        print(f"\nProcessing: {nb_path}")
        data = process_notebook(nb_path)
        if data:
            export_to_csv(data, output_dir, nb_path.stem)
            print(f"  ✓ Extracted metadata")
        else:
            print(f"  ⚠ No experiment_metadata found")


if __name__ == '__main__':
    main()

## How It Works: Step-by-Step

1. **`extract_metadata_from_cell()`**: Uses regex and `eval()` to find and parse a Python dict named `experiment_metadata` from cell source code.

2. **`process_notebook()`**: Loads the `.ipynb` file as JSON, iterates over all code cells, and calls `extract_metadata_from_cell()` on each until it finds a match.

3. **`export_to_csv()`**: Takes the extracted dict and:
   - Identifies nested lists (e.g., `'measurements': [...]`).
   - Uses `pd.json_normalize()` to flatten each list into a table.
   - Saves each table as a separate CSV file (e.g., `exp_name_measurements.csv`).
   - Saves scalar metadata (experiment ID, date, operator) to a separate CSV.

4. **`main()`**: Command-line entry point that scans an input directory for all `.ipynb` files and processes each one.

## Example Usage (in a terminal)

In [ ]:
# Save the extractor script to a file
# python extract_notebook_metadata.py --input-dir ./Jupyter\ Notebook --output-dir ./extracted_data

# Output might look like:
# Scanning ./Jupyter Notebook for .ipynb files...
# Found 2 notebook(s)
#
# Processing: ./Jupyter Notebook/exp_UV_001.ipynb
#   ✓ Extracted metadata
# Exported: ./extracted_data/exp_UV_001_measurements.csv
# Exported: ./extracted_data/exp_UV_001_metadata.csv

## Expected Output Files

After running the extractor on a folder of experiment notebooks, you get:

```
extracted_data/
├── exp_UV_001_measurements.csv
│   (columns: sample, concentration, absorbance, wavelength_nm, experiment_id, date, operator, project, ...)
├── exp_UV_001_metadata.csv
│   (columns: experiment_id, date, operator, project, description, _source_notebook)
├── exp_UV_002_measurements.csv
└── exp_UV_002_metadata.csv
```

These CSV files can then be:
- Imported into a database or LIMS.
- Further processed by analysis scripts.
- Uploaded to a web-based ELN or data repository.
- Used for reporting and auditing (the `_source_notebook` field provides traceability).

## Key Benefits

- **Automatic**: No manual copying or re-entry of data.
- **Traceable**: Each extracted record knows which notebook it came from.
- **Scalable**: Process hundreds of notebooks in seconds.
- **Non-destructive**: Original notebooks are never modified.
- **Flexible**: Easy to customize for different metadata schemas or output formats.
- **Integrable**: CSV output works with any downstream tool (database, Python/R analysis, web APIs).

## Next Steps

1. Create a sample experiment notebook with an `experiment_metadata` cell following the structure shown above.
2. Save the extractor script (e.g., as `extract_notebook_metadata.py` in a `tools/` folder).
3. Run the extractor and verify the CSV outputs.
4. **Optionally extend** the extractor to:
   - Support different metadata variable names (e.g., `lab_record`, `trial_data`).
   - Parse notebook outputs (cells with execution results) for additional data.
   - Push extracted data directly to a database or ELN API instead of CSV.
   - Validate metadata against a JSON schema.